In [ ]:
#!pip install sympy

In [ ]:
from pyomo.environ import *
import numpy as np
from math import pi
from Sympy_QrOPF_ALM_class import SympyACOPFModel
from Sympy_QrOPF_ALM_class import PrintQHDACOPFResults
from Sympy_QrOPF_ALM_class import solve_with_gurobi_from_sympy
from Sympy_QrOPF_ALM_class import initialize_qhd_acopf_log
from Sympy_QrOPF_ALM_class import append_qhd_acopf_results
from qhdopt import QHD
import json
import io
import contextlib

In [ ]:
def load_matpower_json(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    Sbase = float(data["Sbase"])

    # Convert "k1", "k2", ... into integer keys 1, 2, ...
    buses = {int(k.replace("k", "")): v for k, v in data["buses"].items()}
    lines = {int(k.replace("k", "")): v for k, v in data["lines"].items()}
    gens  = {int(k.replace("k", "")): v for k, v in data["gens"].items()}

    return Sbase, buses, lines, gens


In [ ]:
# Initialize the model.
# model = SympyACOPFModel()   # Enable the proximal option if needed later.
n = 2  # Select the test system size: 2, 3, or larger.

if n == 2:
    # 2-bus model
    Sbase = 10.0
    buses = {
        1: [1, 0, 1.00, 0.0, 0.0, 0.0, 0.0, 0.0],
        2: [2, 1, 1.01, 0.0, 0.0, 0.0, 0.3, 0.1],
    }
    lines = {
        1: [1, 2, 0.0452, 0.1852, 0.0204, 1.0, 30.0 / Sbase],
    }
    gens = {
        1: [1, 0.0 / Sbase, 20.0 / Sbase, -20.0 / Sbase, 100.0 / Sbase, 0.00375, 2.0, 0.0],
    }
    model = SympyACOPFModel(Sbase=Sbase, buses=buses, lines=lines, gens=gens)

elif n == 3:
    # 3-bus model
    model = SympyACOPFModel()

else:
    # n-bus model
    Sbase, buses, lines, gens = load_matpower_json(f"case{n}_custom_pretty.json")
    model = SympyACOPFModel(Sbase=Sbase, buses=buses, lines=lines, gens=gens)

print("Model initialized with", n, "buses", model.n_lines, "lines and", model.n_gens, "generators.")


Model initialized with 2 buses 1 lines and 1 generators.


In [ ]:
# =========================
#   Linear ALM + QHD Loop
# =========================

import numpy as np

# 1) Build h(x).
h_func = model.build_h_func()
model.reset_lambdas(0.0)

# 2) Initial point.
xk = model.build_initial_x0()
# xk = np.load("opf_result_ordered.npy", allow_pickle=True)

rho = 5.0
alpha = 5e-3   # Keep the dual step size relatively small.
max_outer = 35
tol = 1e-4

options_str = ["QHD", "Gurobi"]
option = 2  # 1: QHD, 2: Gurobi
qhd_solver = "simbi"  # openjij / simbi

# Initialize the log file.
log_file = initialize_qhd_acopf_log(model, folder="logs", option=option, qhd_solver=qhd_solver)
print("Log file:", log_file)

print("\n===== Start Linear ALM Loop =====\n")

for k in range(max_outer):
    print(f"\n--- Outer Iteration {k} ---")

    Lag, variable_list, Var_bound_list = model.build_linear_ALM_Lagrangian_syms(
        x_center=xk,
        rho=rho,
        ref_bus_id=None,
        mu_prox=10.0
    )

    if option == 2:
        x_new = solve_with_gurobi_from_sympy(
            L_sym=Lag,
            variable_list=variable_list,
            Var_bound_list=Var_bound_list,
            verbose=False
        )
    else:
        raise NotImplementedError("Notebook loop currently supports the Gurobi branch only.")

    lambda_new, h_x = model.update_lambda(x_new, alpha=alpha, h_func=h_func)
    PrintQHDACOPFResults(
        model,
        x_new,
        iteration=k,
        log_file=log_file,
        folder="logs",
        rho=rho,
        alpha=alpha,
        h_x=h_x,
        lambda_vec=lambda_new,
        print_to_console=True,
    )

    max_abs_h = np.max(np.abs(h_x))
    print(f"max |h| = {max_abs_h:.6e}")
    xk = x_new.copy()

    if max_abs_h < tol:
        print("Converged.")
        break


Log file: logs\Buses-2_04-09-2026_15-22-03.txt

===== Start Linear ALM Loop =====


--- Outer Iteration 0 ---

--- Outer Iteration 0 ---
Set parameter Username
Set parameter LicenseID to value 2752566


Update time: 2026-04-09 15:22:03
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	1.008619	-0.000000	1.004798	0.098195	0.060451	0.000000	0.000000
2	0.998554	-0.000000	1.006199	0.000000	0.000000	0.300000	0.100000

Branch Results
LineID	From	To	Pik		Pki		Qik		Qki		LossP		LossQ
1	1	2	0.058209	-0.111079	0.020451	-0.045413	-0.052870	-0.024962

||h(x)|| = 0.280217833934342
[rho-check] ||h_old||=6.685e-01, ||h_new||=2.802e-01, rho=5
Constraint check: False

--- Outer Iteration 1 ---

--- Outer Iteration 1 ---
Update time: 2026-04-09 15:22:04
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	1.009274	-0.000000	1.007714	0.000000	0.036575	0.000000	0.000000
2	0.997222	-0.000000	1.003213	0.000000	0.000000	0.300000	0.100000

Branch Results
LineID	From	To	Pik